# VoxCPM Text-to-Voice (Colab)

Ready-to-run notebook for Reelday.ph marketing voiceover using [VoxCPM](https://github.com/OpenBMB/VoxCPM) (Apache-2.0, free for commercial use).

**Before you run:** set the GPU runtime.
`Runtime` → `Change runtime type` → Hardware accelerator = **T4 GPU** → Save.

Then run the cells top to bottom. Cell 1 (install) takes ~3–5 min on a fresh session and must be re-run each new session.

**Voices:** VoxCPM has no fixed preset list. You either (A) clone your own voice from a short reference clip — section 6, or (B) describe a voice in natural language with Voice Design — section 7.

## 1. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('CUDA available:', torch.cuda.is_available())

## 2. Install VoxCPM (~3–5 min, re-run each session)

In [ ]:
!pip install -q voxcpm soundfile
print('Done. If you see dependency warnings, they are usually safe to ignore.')

## 3. Load the model (downloads weights on first run)

In [ ]:
from voxcpm import VoxCPM
import soundfile as sf
from IPython.display import Audio, display

model = VoxCPM.from_pretrained(
    "openbmb/VoxCPM2",
    load_denoiser=False,
)
print('Model loaded. Sample rate:', model.tts_model.sample_rate)

## 4. Generate voiceover (default model voice)

Edit `TEXT` below, run the cell, then play / download the result.
- `cfg_value`: higher = sticks closer to a reference style (2.0 is a good default).
- `inference_timesteps`: higher = better quality but slower (10 is a good default).

In [ ]:
TEXT = "Capture every moment of your big day with Reelday. Real videographers, edited reels, delivered fast."
OUTPUT_FILE = "voiceover.wav"

wav = model.generate(
    text=TEXT,
    cfg_value=2.0,
    inference_timesteps=10,
)
sf.write(OUTPUT_FILE, wav, model.tts_model.sample_rate)
print('Saved', OUTPUT_FILE)
display(Audio(OUTPUT_FILE))

## 5. Download the audio file

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)

## 6. Use YOUR OWN voice (cloning)

Record or upload a short, clean reference clip (**~5–15 seconds**, no music/noise) of the voice you want to clone. Only clone a voice you own or have permission to use.

**Tip:** for best fidelity, type the exact words spoken in the clip into `PROMPT_TEXT`.

Run the upload cell first, then the cloning cell.

In [ ]:
# Upload your reference voice clip (.wav recommended)
from google.colab import files
uploaded = files.upload()
REF_WAV = list(uploaded.keys())[0]
print('Using reference:', REF_WAV)

In [ ]:
# Exact transcript of what's said in the reference clip (improves fidelity).
PROMPT_TEXT = "Type here exactly what is spoken in your reference clip."

TEXT = "Capture every moment of your big day with Reelday. Book free today."
OUT = "cloned_voiceover.wav"

wav = model.generate(
    text=TEXT,
    prompt_wav_path=REF_WAV,
    prompt_text=PROMPT_TEXT,
    reference_wav_path=REF_WAV,
    cfg_value=2.0,
    inference_timesteps=10,
)
sf.write(OUT, wav, model.tts_model.sample_rate)
print('Saved', OUT)
display(Audio(OUT))
files.download(OUT)

## 6b. Tagalog / Taglish test (in your cloned voice)

Run section 6 first (upload your clip + set `PROMPT_TEXT`). This reuses your `REF_WAV` and `PROMPT_TEXT` and renders several Filipino / Taglish marketing lines so you can judge pronunciation before committing to a full batch.

VoxCPM is tuned mainly on English/Chinese, so listen for odd stress or vowel sounds on the pure-Tagalog lines. Taglish usually fares better.

In [ ]:
TAGALOG_LINES = {
    "tl_1": "Sa Reelday, hindi lang litrato — buong kwento ng kasal mo, naka-video.",
    "tl_2": "Mula sa I do hanggang sa first dance, kami ang kukuha ng bawat sandali.",
    "taglish_1": "Libre mag-start sa Reelday! Real videographers, edited reels, delivered fast.",
    "taglish_2": "Book na ang Reelday mo today — para hindi ka mawalan ng kahit isang moment.",
}

for name, line in TAGALOG_LINES.items():
    wav = model.generate(
        text=line,
        prompt_wav_path=REF_WAV,
        prompt_text=PROMPT_TEXT,
        reference_wav_path=REF_WAV,
        cfg_value=2.0,
        inference_timesteps=10,
    )
    fname = f"{name}.wav"
    sf.write(fname, wav, model.tts_model.sample_rate)
    print(name, '-', line)
    display(Audio(fname))

## 7. Voice Design (describe a voice instead of cloning)

No reference clip needed — describe the voice you want in plain language. This is the closest thing to "picking a voice": just change the description to get a different one. Re-run to get variations, and reuse a description you like for a consistent brand voice.

In [ ]:
VOICE_DESC = "A warm, upbeat young Filipina narrator, friendly and clear, slight excitement."
TEXT = "Your wedding deserves more than photos. With Reelday, every moment becomes a reel."
OUT = "designed_voice.wav"

# VoxCPM2 Voice Design: the voice description is prepended as guidance.
wav = model.generate(
    text=f"[{VOICE_DESC}] {TEXT}",
    cfg_value=2.0,
    inference_timesteps=10,
)
sf.write(OUT, wav, model.tts_model.sample_rate)
print('Saved', OUT)
display(Audio(OUT))
files.download(OUT)

## 8. Batch generation (optional)

Generate several clips at once — useful for multiple ad variations. Add `prompt_wav_path` / `prompt_text` / `reference_wav_path` to each call if you want them in your cloned voice.

In [ ]:
LINES = {
    "hook_1": "Your wedding deserves more than photos.",
    "hook_2": "From I do to the dance floor — we film it all.",
    "cta":    "Book your Reelday today. It's free to start.",
}

for name, line in LINES.items():
    wav = model.generate(text=line, cfg_value=2.0, inference_timesteps=10)
    fname = f"{name}.wav"
    sf.write(fname, wav, model.tts_model.sample_rate)
    print(name)
    display(Audio(fname))